# Notebook 01: Data cleaning & analysis-ready panel

## ABET Student Outcome 6.1 — Quantitative extraction and data quality

**SO 6.1** includes applying sound quantitative methods to real problems. For this project, that means:

- **Traceable inputs**: knowing where the country–year panel came from (merged sources) and what each row represents.
- **Explicit quality checks** before modeling: missing data, key identifiers (`year`, `country`, `iso3c`), and basic sanity checks on ranges.
- **Documented transformations**: imputation rules and the `log1p` target so results are **reproducible** and defensible in your report.

This notebook does **not** re-merge raw CSVs from the World Bank/WHO (that merge can live in a separate script or notebook). It takes the **already-merged** modeling file, cleans it, and writes the **single analysis file** used by Notebooks 02–06.


## What this notebook produces

| Stage | Role |
|-------|------|
| **Input** | One merged country–year file (same schema as your capstone modeling dataset). |
| **Output** | `../data/processed/country_year_modeling_panel_cleaned.csv` — imputed features, `log_mmr`, ready for EDA and models. |

**Run this notebook once** (or whenever the input merge changes) before opening Notebook 02.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Prefer interim "merged" file if you add one; otherwise use project root dataset (same schema)
CANDIDATES = [
    Path("../data/interim/country_year_panel_merged.csv"),
    Path("../data/country_year_modeling_dataset.csv"),
    Path("../data/processed/country_year_modeling_dataset.csv"),
]
INPUT_PATH = next((p for p in CANDIDATES if p.exists()), None)
OUT_PATH = Path("../data/processed/country_year_modeling_panel_cleaned.csv")

if INPUT_PATH is None:
    raise FileNotFoundError(
        "No merged input file found. Expected one of:\n  "
        + "\n  ".join(str(p) for p in CANDIDATES)
    )
print("Loading:", INPUT_PATH.resolve())


In [ ]:
df = pd.read_csv(INPUT_PATH)
print("Shape (rows, columns):", df.shape)
df.head()


## Data quality snapshot (before imputation)

Check missingness and identifiers **before** changing values so SO 6.1 documentation is honest: you can report what was missing and how you addressed it.


In [ ]:
# Required identifiers for panel work
for col in ["year", "country"]:
    if col not in df.columns:
        print(f"WARNING: column '{col}' not found — add or rename for panel logic.")

if "iso3c" in df.columns:
    df["iso3c"] = df["iso3c"].astype(str).str.strip().str.upper()

df["year"] = df["year"].astype(int)

FEATURE_COLS = [
    "gdp_pc", "health_exp_gdp", "fertility", "skilled_births",
    "pm25", "heat_index35_days", "cooling_degree_days",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

miss = df[FEATURE_COLS].isna().sum().sort_values(ascending=False)
print("Missing count per modeling feature (pre-imputation):\n", miss[miss > 0])
if miss.sum() == 0:
    print("No missing values in modeling features.")


## Imputation (within-country medians)

**Rationale:** Filling with a **global** median can blur real differences between countries. Using each country’s own median across years uses that country’s observed history when a value is missing, then a **global** median only where a country still has no usable values (e.g., all NaN for that variable).


In [ ]:
for col in FEATURE_COLS:
    df[col] = df.groupby("country", group_keys=False)[col].transform(
        lambda s: s.fillna(s.median())
    )

df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())

print("Missing in modeling features after imputation:", int(df[FEATURE_COLS].isna().sum().sum()))


## Target: `log_mmr = log1p(mmr)`

MMR is right-skewed; `log1p` stabilizes variance for regression and matches the target used in Notebooks 03+.


In [ ]:
if "log_mmr" not in df.columns:
    df["log_mmr"] = np.log1p(df["mmr"])

df[["mmr", "log_mmr"]].describe()


## Save analysis-ready panel

Downstream notebooks read **only** this file from `../data/processed/`.


In [ ]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH.resolve())
